In [0]:
#adding src path and reading config file
import sys
sys.path.append("/Workspace/DataStore/DataStore/src")
import yaml
from core.Base1 import BaseConfig
from core.Read_Yaml import ReadYaml
from core.Create_Tabls import CreateTable
from core.Create_Schem import CreateSchema
from core.TableHandler import TableHandler
from pyspark.sql.functions import current_timestamp, col, lit
from utils.Utils import Utils
sys.path.append("/Workspace/DataStore/DataStore/src")
path = "/Workspace/DataStore/DataStore/configs/silver.yaml"
settings_path = "/Workspace/DataStore/DataStore/configs/settings.yaml"


In [0]:
#creating schema if not exists
table_info= ReadYaml.read_yaml(path, "Queue")
table_info = BaseConfig(table_info)
CreateSchema.create_schema(spark,table_info.catalog, table_info.schema)

In [0]:
%sql
--creating a managed user silver table
CREATE TABLE IF NOT EXISTS datastore.silver.Queue
(
    ExternalId      STRING NOT NULL,
    QueueName        STRING,
    BusinessunitId  STRING,
    IsUpdated       BOOLEAN,
    CreatedAt       TIMESTAMP,
    UpdatedAt      TIMESTAMP
)USING DELTA;

In [0]:
#transforming bronze data
from pyspark.sql.functions import col, lit, current_timestamp
source_df = (
    spark.table(f"{table_info.catalog}.bronze.{table_info.tableName}")
    .withColumns({
        "IsUpdated": lit(False)
    })
    .select(
        col("id").alias("ExternalId"),
        col("name").alias("QueueName"),
        col("division.id").alias("BusinessunitId"),
        col("InsertedAt").alias("CreatedAt"),
        col("InsertedAt").alias("UpdatedAt"),
    )
)


In [0]:
#getting target silver layer data
target_df = spark.table(f"{table_info.catalog}.silver.{table_info.tableName}")

#generating hash_key to find new and updated records
target_df = target_df.withColumn(
    "hash_key",
    Utils.hash_columns(table_info.hashColumns)
)
source_df = source_df.withColumn(
    "hash_key",
    Utils.hash_columns(table_info.hashColumns)
)

#finding updated records
merge_df = source_df.alias("source").join(
    target_df.alias("target"),
    (
        (col("source.ExternalId") == col("target.ExternalId")) &
        (col("source.hash_key") == col("target.hash_key"))
    ),
    "left_anti"
)

merge_df = merge_df.withColumn("IsUpdated",lit(True))
merge_df = merge_df.drop("hash_key")


In [0]:
#writing to silver layer
from core.TableHandler2 import TableHandler
TableHandler.upsert(spark, merge_df, f"{table_info.catalog}.silver.{table_info.tableName}", key_columns = ["ExternalId"] , exclude_columns = [])
